# TP – Systèmes de recommandation

**Cours** : INF5063 – Machine learning : applications (G. Nollet, L. Benedetti)
**Auteur** : Alban Rouault

Objectif : concevoir et évaluer un système de recommandation de films par filtrage collaboratif
*user-based* sur le jeu de données Kaggle
[The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset).

## Imports et configuration

Toutes les dépendances sont déclarées dans `pyproject.toml` et gérées avec `uv`
(`uv run jupyter lab` pour lancer le notebook).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Affichage des DataFrames
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

# Reproductibilité (séparation train/test, tirages aléatoires)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Emplacement des données
DATA_DIR = Path("Ressources/dataset")

## Exercice 1 – Chargement des données

### 1) Chargement des fichiers

> Récupérer sur Kaggle et charger les fichiers suivants du dataset : `movies_metadata.csv`, `ratings_small.csv` et `links_small.csv`.

In [ ]:
movies = pd.read_csv(DATA_DIR / "movies_metadata.csv", low_memory=False)
ratings = pd.read_csv(DATA_DIR / "ratings_small.csv")
links = pd.read_csv(DATA_DIR / "links_small.csv")

tables = {"movies_metadata": movies, "ratings_small": ratings, "links_small": links}
for name, df in tables.items():
    print(f"{name:<16} {df.shape[0]:>7,} lignes  x {df.shape[1]:>2} colonnes")

**Réponse.** Le dataset [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset)
a été téléchargé depuis Kaggle et décompressé dans `Ressources/dataset/` (non versionné, voir le README).
Les trois fichiers sont chargés avec pandas. `movies_metadata.csv` est lu avec `low_memory=False` car
quelques lignes mal formées mélangent les types dans certaines colonnes (dont `id`) ; le nettoyage est
fait à la question 5.

### 2) Aperçu des tables

> Afficher un aperçu de chaque table et vérifier leur compréhension.

In [ ]:
for name, df in tables.items():
    print(f"── {name} : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
    display(df.head(3))

In [ ]:
def resume_colonnes(df: pd.DataFrame) -> pd.DataFrame:
    """Type, taux de remplissage, cardinalité et exemple pour chaque colonne."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "non nuls": df.notna().sum(),
        "% nuls": (df.isna().mean() * 100).round(1),
        "valeurs distinctes": df.nunique(),
        "exemple": df.iloc[0].astype(str).str.slice(0, 50),
    })

resume_colonnes(movies)

In [ ]:
print("Valeurs possibles de rating :", sorted(ratings["rating"].unique().tolist()))
print("Période des avis :",
      pd.to_datetime(ratings["timestamp"].min(), unit="s").date(), "->",
      pd.to_datetime(ratings["timestamp"].max(), unit="s").date())
print("Doublons (userId, movieId) :", ratings.duplicated(["userId", "movieId"]).sum())
print("Valeurs manquantes dans ratings :", ratings.isna().sum().sum())
print("Valeurs manquantes dans links :", links.isna().sum().to_dict())

**Réponse.**

- **`movies_metadata`** (45 466 lignes, 24 colonnes) : une ligne par film du catalogue TMDB.
  On y trouve l'identifiant TMDB (`id`), l'identifiant IMDB (`imdb_id`), le titre, la date de sortie,
  le budget, les recettes (`revenue`), la durée, la langue, la note moyenne TMDB (`vote_average`) et le
  nombre de votes. Plusieurs colonnes (`genres`, `production_companies`, `belongs_to_collection`…)
  contiennent des listes de dictionnaires stockées sous forme de texte. Certaines colonnes numériques
  (`budget`, `popularity`, `id`) sont lues comme des chaînes à cause de lignes mal formées, ce que
  confirme le tableau de résumé. Des colonnes comme `homepage`, `tagline` ou `belongs_to_collection`
  sont très peu remplies.
- **`ratings_small`** (100 004 lignes, 4 colonnes) : une ligne par avis. `userId` et `movieId` sont
  des identifiants **MovieLens**, `rating` est une note de 0,5 à 5 par pas de 0,5, `timestamp` est une
  date Unix (avis de 1995 à 2016). Aucune valeur manquante, aucun doublon (utilisateur, film) : chaque
  utilisateur n'a noté chaque film qu'une seule fois.
- **`links_small`** (9 125 lignes, 3 colonnes) : une ligne par film du jeu réduit, avec ses trois
  identifiants `movieId` (MovieLens), `imdbId` et `tmdbId`. 13 films n'ont pas de `tmdbId`.

Les avis utilisent donc un espace d'identifiants (MovieLens) différent de celui des métadonnées (TMDB).

### 3) Effectifs

> Combien d'utilisateurs y a-t-il ? De films ? D'avis ?

In [ ]:
n_users = ratings["userId"].nunique()
n_movies_rated = ratings["movieId"].nunique()
n_ratings = len(ratings)
n_movies_links = links["movieId"].nunique()
n_movies_catalog = movies["id"].nunique()

print(f"Utilisateurs                     : {n_users:>7,}")
print(f"Films notés (ratings_small)      : {n_movies_rated:>7,}")
print(f"Films du jeu réduit (links_small): {n_movies_links:>7,}")
print(f"Films du catalogue (metadata)    : {n_movies_catalog:>7,}")
print(f"Avis                             : {n_ratings:>7,}")
print()
print(f"Avis par utilisateur : moyenne {n_ratings / n_users:.0f}, "
      f"médiane {ratings.groupby('userId').size().median():.0f}, "
      f"min {ratings.groupby('userId').size().min()}, "
      f"max {ratings.groupby('userId').size().max()}")
print(f"Densité de la matrice utilisateurs x films : {n_ratings / (n_users * n_movies_rated):.2%}")

**Réponse.** Le jeu réduit contient **671 utilisateurs**, **100 004 avis** et **9 066 films notés**.
Le nombre de films dépend de la table considérée : 9 125 films dans `links_small` (59 n'ont reçu aucun
avis) et 45 466 lignes (45 436 identifiants distincts avant nettoyage) dans `movies_metadata`, qui décrit tout le catalogue TMDB et pas seulement les
films du jeu réduit.

Chaque utilisateur a noté au moins 20 films (médiane 71, un utilisateur en a noté 2 391). La matrice
utilisateurs × films n'est remplie qu'à **1,64 %** : c'est une matrice très creuse, ce qui sera le
principal enjeu du filtrage collaboratif.

### 4) Rôle de `links_small.csv`

> Quelle est l'utilité du fichier `links_small.csv` par rapport aux deux autres fichiers ?

In [ ]:
# Le film movieId = 1 dans ratings : quel est-il ?
tmdb_id = int(links.loc[links["movieId"] == 1, "tmdbId"].iloc[0])
print(f"ratings movieId = 1  ->  links tmdbId = {tmdb_id}  ->  movies_metadata :")
display(movies.loc[movies["id"] == str(tmdb_id), ["id", "imdb_id", "title", "release_date"]])

# Couverture de la table de correspondance
print("movieId de ratings absents de links :", (~ratings["movieId"].isin(links["movieId"])).sum())
print("Films de links sans tmdbId          :", links["tmdbId"].isna().sum())
print("tmdbId de links absents de metadata :",
      (~links["tmdbId"].dropna().astype(int).astype(str).isin(movies["id"])).sum())

**Réponse.** Les deux tables principales n'ont **aucune clé commune** : `ratings_small` identifie
les films par leur `movieId` MovieLens, alors que `movies_metadata` les identifie par leur `id` TMDB
(et leur `imdb_id`). `links_small` est la **table de correspondance** entre ces trois espaces
d'identifiants : c'est elle qui permet de passer d'un avis à un titre de film (et à ses métadonnées),
comme le montre l'exemple ci-dessus (`movieId` 1 → `tmdbId` 862 → *Toy Story*).

Elle sert aussi de **périmètre** : tous les `movieId` notés y figurent, et la version *small* ne
contient que les 9 125 films du jeu réduit, contrairement à `links.csv` qui couvre les 45 000 films.
Ses limites : 13 films n'ont pas de `tmdbId` et 30 `tmdbId` n'ont pas de ligne dans les
métadonnées ; ces films pourront être notés et recommandés, mais pas affichés avec leur titre.

### 5) Nettoyage de `movies_metadata`

> Nettoyer la table `movies_metadata.csv` : convertir tous les ID de films au format numérique (en retirant les films dont les ID ne sont pas numériques), ne conserver qu'une ligne par ID unique.

Le nettoyage se fait en mémoire sur le DataFrame : le fichier CSV d'origine n'est jamais modifié.
On conserve `movies` (brut) et on crée `movies_clean`.

In [ ]:
# 1. ID au format numérique : les valeurs non convertibles deviennent NaN
id_num = pd.to_numeric(movies["id"], errors="coerce")
print(f"Lignes dont l'id n'est pas numérique : {id_num.isna().sum()}")
display(movies.loc[id_num.isna(), ["adult", "budget", "id", "title", "release_date", "popularity", "revenue"]])

Les trois lignes fautives sont des lignes **décalées** : la colonne `adult` contient un morceau de
synopsis, `budget` un chemin d'affiche, `id` une date, et `title` est vide. Elles sont inexploitables.

In [ ]:
# 2. Retrait de ces lignes et conversion en entier
movies_clean = movies.loc[id_num.notna()].copy()
movies_clean["id"] = id_num.loc[id_num.notna()].astype(int)

# 3. Doublons sur l'id
doublons = movies_clean[movies_clean.duplicated("id", keep=False)].sort_values("id")
print(f"Lignes en doublon sur l'id : {doublons['id'].duplicated().sum()} "
      f"({doublons['id'].nunique()} ids concernés, "
      f"{doublons.duplicated().sum()} doublons strictement identiques)")

# Deux exemples : un doublon identique (105045) et un doublon qui diffère (4912)
display(doublons.loc[doublons["id"].isin([105045, 4912]),
                     ["id", "title", "release_date", "popularity", "vote_count", "revenue"]])

# Colonnes qui diffèrent au sein des doublons non identiques
cols_diff = {c for _, g in doublons.groupby("id") for c in g.columns if g[c].astype(str).nunique() > 1}
print("Colonnes qui diffèrent entre doublons :", cols_diff)

In [ ]:
movies_clean = movies_clean.drop_duplicates("id", keep="first").reset_index(drop=True)

print(f"Avant : {len(movies):,} lignes  ->  après : {len(movies_clean):,} lignes")
print("id unique :", movies_clean["id"].is_unique, "| dtype :", movies_clean["id"].dtype)

**Réponse.** Trois lignes ont un `id` non numérique : ce sont des lignes mal formées de l'export
(colonnes décalées, affichées ci-dessus). Elles sont retirées et la colonne `id` est convertie en entier.

Il reste alors 30 lignes en doublon, portant sur 29 films. 17 sont strictement identiques ; les 13 autres
ne diffèrent que par `popularity` (et `vote_count` à une unité près), c'est-à-dire des instantanés du
même film pris à des moments différents. Garder la première occurrence ne perd donc aucune information.

Résultat : **45 433 films**, un par `id`, avec un identifiant entier directement joignable à `tmdbId`
de `links_small`.

### 6) Nettoyage de `ratings_small`

> Nettoyer la table `ratings_small.csv` : retirer les avis manquants et les ID d'utilisateurs et de films non conformes, et convertir tous les ID d'utilisateurs et de films au format numérique.

Critères de conformité retenus :
- avis : `rating` présent, compris entre 0,5 et 5 par pas de 0,5 (échelle MovieLens) ;
- identifiants : numériques, entiers, strictement positifs ;
- un seul avis par couple (utilisateur, film).

In [ ]:
n_avant = len(ratings)
ratings_clean = ratings.copy()

# 1. Avis manquants
manquants = ratings_clean[["userId", "movieId", "rating"]].isna().any(axis=1)
print(f"Avis avec valeur manquante          : {manquants.sum()}")
ratings_clean = ratings_clean[~manquants]

# 2. Identifiants conformes -> entiers
for col in ["userId", "movieId"]:
    num = pd.to_numeric(ratings_clean[col], errors="coerce")
    conforme = num.notna() & (num > 0) & (num % 1 == 0)
    print(f"{col:<8} non conformes               : {(~conforme).sum()}")
    ratings_clean = ratings_clean[conforme].assign(**{col: num[conforme].astype(int)})

# 3. Notes conformes
note_ok = ratings_clean["rating"].between(0.5, 5) & ((ratings_clean["rating"] * 2) % 1 == 0)
print(f"Notes hors échelle MovieLens        : {(~note_ok).sum()}")
ratings_clean = ratings_clean[note_ok]

# 4. Doublons (utilisateur, film)
dup = ratings_clean.duplicated(["userId", "movieId"])
print(f"Doublons (userId, movieId)          : {dup.sum()}")
ratings_clean = ratings_clean[~dup].reset_index(drop=True)

# Films sans métadonnées (conservés : le filtrage collaboratif n'en a pas besoin)
sans_meta = ~ratings_clean["movieId"].isin(
    links.loc[links["tmdbId"].isin(movies_clean["id"]), "movieId"])
print(f"Avis sur un film sans métadonnées   : {sans_meta.sum()} (conservés)")

print(f"\nAvant : {n_avant:,} avis  ->  après : {len(ratings_clean):,} avis")
print(ratings_clean.dtypes.to_string())

**Réponse.** `ratings_small` est déjà propre : aucun avis manquant, aucun identifiant non conforme,
toutes les notes sont sur l'échelle MovieLens et aucun couple (utilisateur, film) n'apparaît deux fois.
Les identifiants étaient déjà lus en entiers par pandas ; le code de nettoyage reste néanmoins
défensif, de façon à fonctionner tel quel sur un fichier moins propre.

194 avis (0,2 %) portent sur des films qu'on ne peut pas relier à `movies_metadata` (13 films sans
`tmdbId` dans `links_small`, 30 `tmdbId` absents du catalogue). On les **conserve** : le filtrage
collaboratif n'utilise que les notes, pas les métadonnées. Seul leur titre manquera à l'affichage.

Les **100 004 avis** sont donc tous gardés.

### 7) Les 10 films les plus anciens

> Déterminer les 10 films les plus anciens du dataset.

In [ ]:
# Conversion de release_date en date ; les valeurs invalides deviennent NaT
movies_clean["release_date"] = pd.to_datetime(movies_clean["release_date"], errors="coerce")

print("Films sans date de sortie :", movies_clean["release_date"].isna().sum())
futur = movies_clean[movies_clean["release_date"] > "2017-12-31"]
print(f"Films datés après 2017 (extraction du dataset) : {len(futur)}")
display(futur[["id", "title", "release_date", "status", "vote_count"]])

In [ ]:
cols = ["id", "title", "release_date", "runtime", "original_language", "vote_count"]
plus_anciens = movies_clean.dropna(subset=["release_date"]).nsmallest(10, "release_date")
plus_anciens[cols]

In [ ]:
# Même question restreinte aux films effectivement notés dans ratings_small
tmdb_notes = links.loc[links["movieId"].isin(ratings_clean["movieId"]), "tmdbId"].dropna().astype(int)
films_notes = movies_clean[movies_clean["id"].isin(tmdb_notes)]
print(f"Films notés retrouvés dans les métadonnées : {len(films_notes):,}")
films_notes.nsmallest(10, "release_date")[cols]

**Réponse.** Après conversion de `release_date` en date (87 films n'en ont pas, et 6 sont datés
de 2018 ou 2020 : des films annoncés, non sortis au moment de l'extraction du dataset en 2017),
les dix films les plus anciens du catalogue sont des **films des débuts du cinéma, de 1874 à 1890**,
d'une minute chacun : *Passage of Venus* (1874), *Sallie Gardner at a Gallop* (1878),
*Buffalo Running* (1883), *Man Walking Around a Corner* (1887), etc. Ce sont des expériences de
chronophotographie plutôt que des films au sens moderne, mais les dates sont exactes.

Si l'on se restreint aux films réellement notés dans `ratings_small` (9 025 films retrouvés),
le plus ancien est *A Trip to the Moon* (1902), suivi de *The Birth of a Nation* (1915) et de
films muets des années 1916 à 1921.

### 8) Les 10 plus gros succès au box-office

> Déterminer les 10 films ayant eu le plus gros succès au box-office (colonne `revenue`).

In [ ]:
movies_clean["revenue"] = pd.to_numeric(movies_clean["revenue"], errors="coerce")
rev = movies_clean["revenue"]

print(f"revenue manquant (NaN)      : {rev.isna().sum():>6,}")
print(f"revenue = 0                 : {(rev == 0).sum():>6,}  ({(rev == 0).mean():.1%} des films)")
print(f"0 < revenue < 1 000         : {rev.between(1, 999).sum():>6,}")
print(f"revenue >= 1 000            : {(rev >= 1000).sum():>6,}")

# Exemples de valeurs suspectes : quelques dollars de recettes pour des films distribués en salle
display(movies_clean.loc[rev.between(1, 999), ["title", "release_date", "budget", "revenue"]].head(5))

In [ ]:
top_revenue = movies_clean.nlargest(10, "revenue").copy()
top_revenue["revenue (M$)"] = (top_revenue["revenue"] / 1e6).round(0)
top_revenue["budget (M$)"] = (pd.to_numeric(top_revenue["budget"], errors="coerce") / 1e6).round(0)
top_revenue[["id", "title", "release_date", "revenue (M$)", "budget (M$)", "vote_average", "vote_count"]]

**Réponse.** La colonne `revenue` est très incomplète : **38 032 films (84 %) ont une recette de 0**,
qui code en réalité une valeur inconnue, et 154 films affichent quelques dollars ou centaines de
dollars, ce qui est manifestement une unité erronée (millions) ou une saisie fausse. Ces anomalies
n'affectent pas le classement des plus gros succès, qui ne dépend que des valeurs élevées.

Le top 10 est dominé par des blockbusters récents : **Avatar** (2,79 milliards de dollars),
**Star Wars : The Force Awakens** (2,07), **Titanic** (1,85), **The Avengers** (1,52),
**Jurassic World** (1,51), **Furious 7** (1,51), **Avengers : Age of Ultron** (1,41),
**Harry Potter and the Deathly Hallows – Part 2** (1,34), **Frozen** (1,27) et
**Beauty and the Beast** (1,26). Les recettes sont en dollars courants, non corrigées de l'inflation,
ce qui favorise mécaniquement les films sortis après 2009.